In [ ]:
# make sure that the needed packages are installed beforehand! if need be, uncomment and run:
# !pip install numpy pandas scipy mat73

import numpy as np
import pandas as pd
import scipy.io as sio
import mat73

def load_mat_any(path):
    try:
        mat = mat73.loadmat(path)
    except TypeError:
        # fall back to scipy in case
        mat = sio.loadmat(path, simplify_cells=True)
    return mat

In [ ]:
MAT_PATH = "P1_S1_EMG.mat"
META_JSON_PATH = "P1_S1_meta.json"

mat = load_mat_any(MAT_PATH)
data = mat['miDB']

# confirming shape
print(type(data))
if isinstance(data, list):
    print('n_trials:', len(data))
    print('fields:', list(data[0].keys()))
else:
    print('fields:', list(data.keys()))

In [ ]:
def normalize_struct_array(data):
    if isinstance(data, list):
        return data
    elif isinstance(data, dict):
        # struct-of-arrays -> list-of-dicts
        keys = list(data.keys())
        n = len(data[keys[0]])
        return [{k: data[k][i] for k in keys} for i in range(n)]
    else:
        raise TypeError(f'Unexpected type for data: {type(data)}')

trials = normalize_struct_array(data)
print('n_trials:', len(trials))
print('fields:', list(trials[0].keys()))

In [ ]:
#metadata dataframe
meta_cols = ['trialID', 'TaskNumber', 'TrialNumber', 'RestTime', 'HoldTime', 'onset_idx']
trial_meta = pd.DataFrame([{k: trial[k] for k in meta_cols} for trial in trials])
trial_meta.head()

In [ ]:
# Build the long-format EMG DataFrame

#Check `emg.shape` for the first trial before running on all trials.
#if it's `nChannels x nSamples` instead of `nSamples x nChannels`, transpose with `.T`.

In [ ]:
# sanity check on array orientation
emg0 = np.asarray(trials[0]['EMG1k'])
print('emg1k shape (trial 0):', emg0.shape)
# expected: (nSamples, nChannels)

In [ ]:
TRANSPOSE = False  # set True if the shape above is (nChannels, nSamples)

rows = []
for trial in trials:
    tid = trial['trialID']
    emg = np.asarray(trial['EMG1k'])
    emgf = np.asarray(trial['EMG1kf'])
    if TRANSPOSE:
        emg = emg.T
        emgf = emgf.T
    n_samples, n_channels = emg.shape

    rows.append(pd.DataFrame({
        'trialID': tid,
        'sample': np.repeat(np.arange(n_samples), n_channels),
        'channel': np.tile(np.arange(n_channels), n_samples),
        'EMG1k': emg.flatten(),
        'EMG1kf': emgf.flatten(),
    }))

emg_df = pd.concat(rows, ignore_index=True)
emg_df.head()

In [ ]:
# add JSON metadata (movementNumber, onsetIDX)

In [ ]:
meta_json = pd.read_json(META_JSON_PATH)

trial_meta['trialID'] = trial_meta['trialID'].astype(int)
meta_json['trialID'] = meta_json['trialID'].astype(int)
emg_df['trialID'] = emg_df['trialID'].astype(int)

trial_meta = trial_meta.merge(
    meta_json[['trialID', 'movementNumber', 'onsetIDX']],
    on='trialID',
    how='left'
)

emg_df = emg_df.merge(trial_meta[['trialID', 'onsetIDX', 'movementNumber']], on='trialID')

trial_meta.head()

In [ ]:
emg_df.head()